# S2F — Few-Shot Segmentation (Colab)

Pipeline: sam encoder → **ProtoFormer** (rgb+topografi, fusi weight-shared) → skip connection.

**Workflow baru:** semua settingan training (shot, epochs, batch, lr, topografi, path default, checkpoint, blacklist, output) ada di **`train.sh`**.
Notebook ini hanya:
1. set **path** (Drive) + daftar run,
2. sync repo, install deps,
3. siapkan data (PASCAL opsional),
4. jalankan `bash train.sh ...`,
5. evaluasi lewat `bash train.sh ... --test-only`.

Dataset default: **landslide** (multimodal, folder `3.TRAINING` / `4.VALIDATION` / `5.TESTING`).
Set `DATASET = "pascal"` untuk protokol PASCAL-5i.

## 1. Konfigurasi path & daftar run

Hyperparameter **tidak** diisi di sini — cukup path dan run mana yang dijalankan.
Kalau butuh override, isi `EXTRA_ARGS` (diteruskan apa adanya ke `train.sh`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR    = "/content/drive/MyDrive/project/S2F"
REPO_URL    = "https://github.com/USER/REPO.git"   # WAJIB diisi
REPO_BRANCH = "master"

# ---- pilih dataset: "landslide" atau "pascal" ----
DATASET = "landslide"

# ---- path dataset landslide (default = struktur Google Drive kamu) ----
TRAIN_ROOT = "/content/drive/MyDrive/project/3.TRAINING"
VAL_ROOT   = "/content/drive/MyDrive/project/4.VALIDATION"
TEST_ROOT  = "/content/drive/MyDrive/project/5.TESTING"
BLACKLIST  = "/content/drive/MyDrive/project/black_list.txt"
HIERA_CKPT = "/content/drive/MyDrive/project/checkpoint/sam2_hiera_large.pt"
OUT_DIR    = "/content/drive/MyDrive/project/runs"

# ---- parameter PASCAL (hanya dipakai kalau DATASET == "pascal") ----
DATA_ROOT_LOCAL = "/content/datasets"
ARCHIVE_DIR     = "/content/drive/MyDrive/project/datasets/_archives"
SPLIT_ROOT      = "/content/drive/MyDrive/project/datasets/splits/pascal"
CACHE_DIR       = "/content/drive/MyDrive/project/datasets/_cache"
FOLDS           = [0, 1, 2, 3]

VOC_DIR  = f"{DATA_ROOT_LOCAL}/VOC2012"
SBD_DIR  = f"{DATA_ROOT_LOCAL}/SBD"
SBD_IMG  = f"{SBD_DIR}/img"
SBD_CLS  = f"{SBD_DIR}/cls"
SBD_PNG  = f"{SBD_DIR}/cls_png"
VOC_IMG  = f"{VOC_DIR}/JPEGImages"
VOC_MASK = f"{VOC_DIR}/SegmentationClass"

# ---- override opsional untuk semua run (kosongkan = pakai default train.sh) ----
EXTRA_ARGS = []   # mis: ["--shot","5","--epochs","30"]

# ---- daftar run ----
if DATASET == "landslide":
    RUNS = [
        ["--dataset", "landslide",
         "--train-root", TRAIN_ROOT, "--val-root", VAL_ROOT, "--test-root", TEST_ROOT,
         "--blacklist", BLACKLIST, "--hiera-path", HIERA_CKPT, "--output", OUT_DIR,
         "--feature", "dtm"],
        # contoh run kedua (5-shot, fusi DTM+HILLSHADE):
        # ["--dataset","landslide","--train-root",TRAIN_ROOT,"--val-root",VAL_ROOT,
        #  "--test-root",TEST_ROOT,"--blacklist",BLACKLIST,"--hiera-path",HIERA_CKPT,
        #  "--output",OUT_DIR,"--topo","DTM","HILLSHADE","--shot","5"],
    ]
    RUN_LABELS = [f"landslide-run{i}" for i in range(len(RUNS))]
else:
    RUNS, RUN_LABELS = [], []
    for f in FOLDS:
        RUNS.append([
            "--dataset", "pascal", "--fold", str(f),
            "--train-img-dir", SBD_IMG, "--train-mask-dir", SBD_PNG,
            "--train-split", f"{SPLIT_ROOT}/fold{f}/train.txt",
            "--val-img-dir", VOC_IMG, "--val-mask-dir", VOC_MASK,
            "--val-split", f"{SPLIT_ROOT}/fold{f}/val.txt",
            "--test-img-dir", VOC_IMG, "--test-mask-dir", VOC_MASK,
            "--test-split", f"{SPLIT_ROOT}/fold{f}/val.txt",
            "--hiera-path", HIERA_CKPT, "--output", OUT_DIR,
            "--cache-dir", CACHE_DIR,
        ])
        RUN_LABELS.append(f"fold{f}")

print(f"config ok | dataset={DATASET} | runs={len(RUNS)}")
for lbl, run in zip(RUN_LABELS, RUNS):
    print(" -", lbl, ":", " ".join(run))

## 1b. Ambil / perbarui kode dari git

In [ ]:
import os, subprocess

def git(*a):
    print("$", "git", *a)
    subprocess.run(["git", *a], check=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    if os.path.isdir(REPO_DIR) and os.listdir(REPO_DIR):
        print(f"{REPO_DIR} ada tapi bukan git repo -> lewati clone.")
        print("Pindahkan/rename dulu, atau set REPO_DIR ke folder kosong.")
    else:
        git("clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR)
else:
    git("-C", REPO_DIR, "fetch", "origin", REPO_BRANCH)
    git("-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}")

subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"])

## 2. Setup environment
Ekstensi CUDA SAM2 (`_C`) tidak dipakai pipeline ini, jadi tidak perlu di-build.

In [ ]:
!pip -q install hydra-core==1.3.2 omegaconf imageio pysodmetrics tqdm scipy timm opencv-python

In [ ]:
import os, sys
assert os.path.isdir(REPO_DIR), f"REPO_DIR tidak ada: {REPO_DIR}"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

assert os.path.isfile(os.path.join(REPO_DIR, "train.sh")), "train.sh tidak ditemukan"
check_paths = [TRAIN_ROOT, VAL_ROOT, BLACKLIST, HIERA_CKPT] if DATASET == "landslide" else [SBD_IMG, VOC_IMG, VOC_MASK, HIERA_CKPT]
for p in check_paths:
    print("OK " if os.path.exists(p) else "MISSING ", p)

## 3. Siapkan data

- **landslide**: data diasumsikan sudah ada di `TRAIN_ROOT` / `VAL_ROOT` / `TEST_ROOT` (Drive). Tiap folder berisi `image/`, `dtm/`, `hillshade/`, `label/` (case-insensitive).
- **pascal**: unduh + ekstrak + konversi SBD + buat split (sama seperti sebelumnya).

In [ ]:
if DATASET == "pascal":
    get_ipython().system('python download_data.py --archive_dir "$ARCHIVE_DIR" --extract_root "$DATA_ROOT_LOCAL" --ckpt_path "$HIERA_CKPT"')
    if not os.path.isdir(SBD_PNG) or len(os.listdir(SBD_PNG)) == 0:
        get_ipython().system('python prepare_sbd.py --cls_dir "$SBD_CLS" --out_dir "$SBD_PNG"')
    else:
        print("SBD_PNG sudah ada, lewati")
    if not os.path.isdir(f"{SPLIT_ROOT}/fold0"):
        get_ipython().system('python make_splits.py --sbd_img_dir "$SBD_IMG" --sbd_mask_dir "$SBD_PNG" --voc_img_dir "$VOC_IMG" --voc_mask_dir "$VOC_MASK" --out_dir "$SPLIT_ROOT"')
    else:
        print("split sudah ada, lewati")
else:
    for p in [TRAIN_ROOT, VAL_ROOT, TEST_ROOT]:
        print("OK " if os.path.isdir(p) else "MISSING ", p)
    print("OK " if os.path.isfile(BLACKLIST) else "MISSING ", BLACKLIST)
    print("catatan: isi tiap split = image/ dtm/ hillshade/ label/ (case-insensitive)")

## 4. Training (satu run per pemanggilan train.sh)

Semua hyperparameter diambil dari `train.sh`. Notebook hanya meneruskan path/run.
Tambah `--test` kalau ingin evaluasi langsung menempel setelah training (opsional; sel 5 sudah terpisah).

In [ ]:
import subprocess

for label, run in zip(RUN_LABELS, RUNS):
    cmd = ["bash", "train.sh", *run, *EXTRA_ARGS]
    print("=" * 70)
    print(f"TRAIN {label}")
    print("=" * 70)
    print(">>", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

## 5. Evaluasi (mIoU, FB-IoU) via `train.sh --test-only`

`--test-only` memakai settingan & nama checkpoint yang sama dengan training (train.sh yang menentukan `save_path`).

In [ ]:
import subprocess, re
import numpy as np

rows = []
for label, run in zip(RUN_LABELS, RUNS):
    cmd = ["bash", "train.sh", *run, *EXTRA_ARGS, "--test-only"]
    print("\n" + "=" * 70)
    print(f"TEST {label}")
    print("=" * 70)
    print(">>", " ".join(cmd), flush=True)
    out = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
    print(out.stdout)
    if out.returncode != 0:
        print(out.stderr[-2000:])
    m = re.search(r"mIoU\s*:\s*([\d.]+)", out.stdout)
    f = re.search(r"FB-IoU\s*:\s*([\d.]+)", out.stdout)
    if m and f:
        rows.append((label, float(m.group(1)), float(f.group(1))))

print("\n================ HASIL AKHIR ================")
for label, miou, fb in rows:
    print(f"{label}: mIoU {miou:.4f}  FB-IoU {fb:.4f}")
if rows:
    print(f"mean ({len(rows)} run): mIoU {np.mean([r[1] for r in rows]):.4f}  "
          f"FB-IoU {np.mean([r[2] for r in rows]):.4f}")